In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew
from scipy.special import boxcox1p
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Lasso, ElasticNet, Ridge
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings

warnings.filterwarnings('ignore')

# 1. 加载数据
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 2. 离群点处理
train = train.drop(train[(train['GrLivArea']>4000) & (train['SalePrice']<300000)].index)

# 3. 目标变量处理
y = np.log1p(train['SalePrice'])
train_id = train['Id']
test_id = test['Id']
all_data = pd.concat((train, test)).drop(['Id', 'SalePrice'], axis=1)

# ---------------------------------------------------------
# 4. 严格的数据清洗
# ---------------------------------------------------------

# A. 明确数值列名单
num_cols = all_data.select_dtypes(include=[np.number]).columns.tolist()

# B. 明确类别列名单
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()

# C. 对数值列：强制转数字，报错变NaN，最后填补0
for col in num_cols:
    all_data[col] = pd.to_numeric(all_data[col], errors='coerce').fillna(0)

# D. 对类别列：填充 "None"
for col in cat_cols:
    all_data[col] = all_data[col].fillna("None")

# E. 统计学补充
all_data['MSZoning'] = all_data['MSZoning'].replace("None", all_data['MSZoning'].mode()[0])
all_data['Functional'] = all_data['Functional'].replace("None", "Typ")
all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))

# ---------------------------------------------------------
# 5. 特征构造
# ---------------------------------------------------------
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBath'] = all_data['FullBath'] + (0.5 * all_data['HalfBath']) + \
                        all_data['BsmtFullBath'] + (0.5 * all_data['BsmtHalfBath'])

# ---------------------------------------------------------
# 6. 处理偏度
# ---------------------------------------------------------
# 重新获取最新的数值特征索引，确保不包含任何 object 类型
final_numeric_feats = all_data.select_dtypes(include=[np.number]).columns

# 重点：在 apply skew 之前再次确保没有 strings
skewed_feats = all_data[final_numeric_feats].apply(lambda x: skew(x)).sort_values(ascending=False)

high_skew = skewed_feats[abs(skewed_feats) > 0.75]
skewed_features = high_skew.index
lam = 0.15
for feat in skewed_features:
    all_data[feat] = boxcox1p(all_data[feat], lam)

# ---------------------------------------------------------
# 7. 建模与预测
# ---------------------------------------------------------
all_data = pd.get_dummies(all_data)
X_train = all_data[:len(y)]
X_test = all_data[len(y):]

lasso = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, random_state=42))
ridge = make_pipeline(RobustScaler(), Ridge(alpha=15))
gbr = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.05, max_depth=4, 
                                 max_features='sqrt', min_samples_leaf=15, 
                                 min_samples_split=10, loss='huber', random_state=42)
xgb = XGBRegressor(learning_rate=0.05, n_estimators=2200, max_depth=3, random_state=42)
lgbm = LGBMRegressor(objective='regression', num_leaves=5, learning_rate=0.05, n_estimators=720)

print("正在训练融合模型...")
lasso.fit(X_train, y)
ridge.fit(X_train, y)
gbr.fit(X_train, y)
xgb.fit(X_train, y)
lgbm.fit(X_train, y)

# 加权平均
y_pred = (0.1 * lasso.predict(X_test) + 
          0.1 * ridge.predict(X_test) + 
          0.2 * gbr.predict(X_test) + 
          0.3 * xgb.predict(X_test) + 
          0.3 * lgbm.predict(X_test))

final_price = np.expm1(y_pred)
submission = pd.DataFrame({'Id': test_id, 'SalePrice': final_price})
submission.to_csv('submission_v2.csv', index=False)
print("保存成功！")